In [ ]:
import os
import re
import json
import csv

def process_ispy_file(input_path, output_dir):
    """
    Procesa un archivo de datos iSpy WebGL:
      - Reemplaza 'nan' → null
      - Carga el JSON
      - Separa cada Type en un CSV distinto
      - Expande campos v3d en columnas: campo_x, campo_y, campo_z
    """
    print(f"Procesando: {input_path}")

    # Leer archivo original
    with open(input_path, 'r') as f:
        text = f.read()

    # Reemplazar 'nan' que rompe el JSON
    text_clean = re.sub(r'\bnan\b', 'null', text)

    # Cargar JSON
    try:
        data = json.loads(text_clean)
    except Exception as e:
        print(f" Error al leer {input_path}: {e}")
        return

    types = data.get("Types", {})
    collections = data.get("Collections", {})

    # Crear carpeta de salida
    os.makedirs(output_dir, exist_ok=True)

    # Crear un CSV por cada type
    for name, fields in types.items():
        rows = collections.get(name, [])

        output_file = os.path.join(output_dir, f"{name}.csv")

        # Construir encabezados con v3d bien nombrados
        header = []
        for field_name, field_type in fields:
            if field_type == "v3d":
                header += [
                    f"{field_name}_x",
                    f"{field_name}_y",
                    f"{field_name}_z"
                ]
            else:
                header.append(field_name)

        with open(output_file, "w", newline='') as f:
            writer = csv.writer(f)
            writer.writerow(header)

            for row in rows:
                flat = []
                # Recorremos cada valor junto con su tipo
                for (val, (field_name, field_type)) in zip(row, fields):

                    if field_type == "v3d":
                        # Expandir lista [x, y, z]
                        x, y, z = val
                        flat += [x, y, z]

                    elif isinstance(val, list):
                        # Si es otra lista compleja → guardarla como texto
                        flat.append(json.dumps(val))

                    else:
                        flat.append(val)

                writer.writerow(flat)

        print(f"  ✔ CSV creado: {output_file}")


def process_all_files(input_folder, output_base_folder):
    """
    Procesa todos los archivos de la carpeta input_folder que sean JSON de iSpy.
    """
    for filename in os.listdir(input_folder):
        path = os.path.join(input_folder, filename)

        if not os.path.isfile(path):
            continue

        print(f"Encontrado archivo: {filename}")

        # Carpeta de salida por archivo
        output_folder = os.path.join(output_base_folder, os.path.splitext(filename)[0])
        process_ispy_file(path, output_folder)


# ---------------------------------------------------------------------
# USO:
# ---------------------------------------------------------------------
process_all_files(
    input_folder="/home/debian12/Documentos/UD/Mineria_de_Datos/Parcial Final/Events/Run_198272",
    output_base_folder="/home/debian12/Documentos/UD/Mineria_de_Datos/Parcial Final/Events/Run_198272/CSV"
)
# ---------------------------------------------------------------------


Encontrado archivo: Event_272621358
Procesando: /home/debian12/Documentos/UD/Mineria_de_Datos/Parcial Final/Events/Run_198272/Event_272621358
  ✔ CSV creado: /home/debian12/Documentos/UD/Mineria_de_Datos/Parcial Final/Events/Run_198272/CSV/Event_272621358/Event_V2.csv
  ✔ CSV creado: /home/debian12/Documentos/UD/Mineria_de_Datos/Parcial Final/Events/Run_198272/CSV/Event_272621358/Products_V1.csv
  ✔ CSV creado: /home/debian12/Documentos/UD/Mineria_de_Datos/Parcial Final/Events/Run_198272/CSV/Event_272621358/EBRecHits_V2.csv
  ✔ CSV creado: /home/debian12/Documentos/UD/Mineria_de_Datos/Parcial Final/Events/Run_198272/CSV/Event_272621358/EERecHits_V2.csv
  ✔ CSV creado: /home/debian12/Documentos/UD/Mineria_de_Datos/Parcial Final/Events/Run_198272/CSV/Event_272621358/ESRecHits_V2.csv
  ✔ CSV creado: /home/debian12/Documentos/UD/Mineria_de_Datos/Parcial Final/Events/Run_198272/CSV/Event_272621358/HBRecHits_V2.csv
  ✔ CSV creado: /home/debian12/Documentos/UD/Mineria_de_Datos/Parcial Final/E